# LG Aimers — 0816 Domain v1

목표:

1. 기존 SOTA 계열 전처리 유지
   - base features
   - strict season Flat History
   - Current Season Decomposition
   - pitcher drift / safe context
   - HLogit
   - `minus_batter_flat` 방향 유지
2. Domain v1 피처 추가
   - Count pressure
   - Runner / stretch
   - Game pressure
   - Matchup
   - Recent trend
   - Sample-size reliability
   - Pitcher role × inning
3. 최근 3개 시즌 Rolling CV
4. Group ablation
5. Individual ablation
6. 최종 CatBoost 재학습 및 모델/메타데이터/history 저장

## 누수 방지 원칙

- 모든 history 집계 피처는 **`season < target_season`** 만 사용합니다.
- pitcher role lookup 역시 대상 시즌 이전 데이터만 사용합니다.
- test 내부 행끼리 집계하지 않습니다.
- `trackman_history.csv`는 이 노트북에서 사용하지 않습니다.
- `0816_SOTA.ipynb`의 latent-skill 모델은 추론용 모델만 있고 fold별 cross-fit 학습 레시피가 확인되지 않으므로,
  **Domain v1 rolling CV에서는 의도적으로 제외**합니다.  
  나중에 latent skill을 넣으려면 validation 시즌마다 `season < valid_season`으로 skill 모델을 별도 재학습해야 합니다.

In [1]:
# ============================================================
# 0. Imports
# ============================================================

import gc
import json
import os
import time
import sys
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

try:
    from catboost import CatBoostClassifier
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.8"]
    )
    from catboost import CatBoostClassifier

from IPython.display import display

ID = "row_id"
TARGET = "control_success"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [2]:
# ============================================================
# 1. Paths / experiment config
# ============================================================

# 수정 필요 시 이 경로만 바꾸면 됩니다.
PROJECT_DIR = Path(os.environ.get("lgaimers", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output" / "0816_domain_v1"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"  # feature-schema 교집합 확인용. 없어도 학습 가능.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# Feature-engineering params
# -----------------------
FLAT_ALPHA = 20.0
CONTEXT_EFFECT_ALPHA = 20.0
CURRENT_SEASON_ALPHA = 100.0
LOGIT_EPS = 1e-5

# Role heuristic
ROLE_MIN_PITCHES = 100
ROLE_STARTER_EARLY_SHARE = 0.45
ROLE_STARTER_MEAN_INNING_MAX = 5.5
ROLE_CLOSER_LATE_SHARE = 0.55

# -----------------------
# CV / ablation params
# -----------------------
N_ROLLING_FOLDS = 3
EARLY_STOPPING_ROUNDS = 100

RUN_GROUP_ABLATION = True
RUN_INDIVIDUAL_ABLATION = True

# True면 모든 Domain v1 개별 피처를 3-fold rolling으로 평가합니다.
# 시간이 너무 오래 걸리면 False로 두고 group ablation까지만 먼저 확인하세요.
FULL_ROLLING_INDIVIDUAL_ABLATION = True

ABLATION_IMPROVEMENT_TOL = 1e-5
INDIVIDUAL_IMPROVEMENT_TOL = 1e-5

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)

PROJECT_DIR : /Users/chunyoomin/lgaimers
DATA_DIR    : /Users/chunyoomin/lgaimers/data
OUTPUT_DIR  : /Users/chunyoomin/lgaimers/output/0816_domain_v1


In [4]:
# ============================================================
# 2. Load train / optional test
# ============================================================

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"train.csv를 찾을 수 없습니다: {TRAIN_PATH}\n"
        "PROJECT_DIR 또는 DATA_DIR를 수정하세요."
    )

train_raw = pd.read_csv(TRAIN_PATH)

test_raw = None
if TEST_PATH.exists():
    test_raw = pd.read_csv(TEST_PATH)
    print(f"test loaded: {test_raw.shape}")
else:
    print("[WARN] test.csv 없음 — train-only feature schema로 진행합니다.")

required_train = {
    ID,
    TARGET,
    "season",
    "pitcher_id",
    "balls_before",
    "strikes_before",
    "outs_before",
    "base_state",
    "pitcher_hand",
    "batter_hand",
    "top_bottom",
    "inning",
    "asof_pitcher_n",
    "asof_pitcher_success_rate",
    "asof_pitcher_middle_rate",
    "asof_pitcher_reverse_rate",
    "asof_pitcher_ball_rate",
    "asof_pitcher_strike_rate",
    "asof_pitcher_pitchmix_n",
    "asof_pitcher_fastball_rate",
    "asof_pitcher_breaking_rate",
    "asof_pitcher_offspeed_rate",
    "asof_pitcher_prev1_game_success_rate",
    "asof_pitcher_prev3_game_success_rate",
    "asof_pitcher_prev5_game_success_rate",
}

missing_required = sorted(required_train.difference(train_raw.columns))
if missing_required:
    raise KeyError(f"train.csv 필수 컬럼 누락: {missing_required}")

print("train shape:", train_raw.shape)
print("seasons:", sorted(train_raw["season"].dropna().unique().tolist()))
print("target mean:", float(train_raw[TARGET].mean()))

test loaded: (5, 48)
train shape: (1475092, 49)
seasons: [2019, 2020, 2021, 2022, 2023, 2024]
target mean: 0.5237659752747625


## 3. 기존 SOTA 계열 Base Features

`0816_SOTA`의 추론 전처리와 맞추는 기본 축입니다.

- `count_state`
- `base_out_state`
- `hand_matchup`
- pitcher home / pitcher-team win expectancy
- history missing
- prev1/3/5 vs career delta
- 표본 수 × 누적률(`*_count500`)

In [5]:
# ============================================================
# 3. Existing base features
# ============================================================

def add_base_features(df):
    df = df.copy()

    df["count_state"] = (
        df["balls_before"].astype("Int64").astype(str)
        + "-"
        + df["strikes_before"].astype("Int64").astype(str)
    )

    df["base_out_state"] = (
        df["base_state"].astype(str)
        + "_"
        + df["outs_before"].astype("Int64").astype(str)
    )

    df["hand_matchup"] = (
        df["pitcher_hand"].astype(str)
        + "_"
        + df["batter_hand"].astype(str)
    )

    df["pitcher_is_home"] = df["top_bottom"].eq("T").astype("int8")

    if {"home_win_expectancy", "away_win_expectancy"}.issubset(df.columns):
        df["pitcher_team_win_expectancy"] = np.where(
            df["pitcher_is_home"].eq(1),
            df["home_win_expectancy"],
            df["away_win_expectancy"],
        ).astype("float32")
    else:
        df["pitcher_team_win_expectancy"] = np.nan

    df["pitcher_history_missing"] = (
        df["asof_pitcher_n"].fillna(0).eq(0).astype("int8")
    )

    df["previous_game_history_missing"] = (
        df["asof_pitcher_prev1_game_success_rate"]
        .isna()
        .astype("int8")
    )

    for window in (1, 3, 5):
        df[f"pitcher_success_prev{window}_delta"] = (
            df[f"asof_pitcher_prev{window}_game_success_rate"]
            - df["asof_pitcher_success_rate"]
        ).astype("float32")

    pitcher_n_cap500 = (
        pd.to_numeric(df["asof_pitcher_n"], errors="coerce")
        .fillna(0)
        .clip(upper=500)
        .astype("float32")
    )

    for metric in ("success", "reverse", "ball", "strike"):
        col = f"asof_pitcher_{metric}_rate"
        df[f"pitcher_{metric}_count500"] = (
            pd.to_numeric(df[col], errors="coerce").astype("float32")
            * pitcher_n_cap500
        ).astype("float32")

    return df

## 4. Domain v1 Features

### Count pressure
- pitcher ahead / even / batter ahead
- 0-2, 3-0, two-strike, three-ball, full-count
- 선택적 count × pitcher tendency interaction

### Runner / stretch
- 주자 수
- stretch 여부
- 1루 단독
- 득점권
- 3루 주자 + 2아웃 미만

### Game pressure
- 투수 팀 기준 점수차 절댓값
- blowout(6점 이상)
- late & close(7회 이상, 3점 이내)

### Matchup
- same-side

### Recent trend
- prev1-prev5
- prev3-prev5
- prev1/3/5 range

### Sample size
- log1p(asof_pitcher_n)

Pitcher role은 별도 strict-history lookup으로 뒤에서 추가합니다.

In [6]:
# ============================================================
# 4. Domain v1 static features
# ============================================================

PITCHER_AHEAD_COUNTS = {
    (0, 1), (0, 2), (1, 2),
}
BATTER_AHEAD_COUNTS = {
    (1, 0), (2, 0), (2, 1), (3, 0), (3, 1),
}


def _numeric_or_zero(df, column):
    if column not in df.columns:
        return pd.Series(0.0, index=df.index, dtype="float32")
    return (
        pd.to_numeric(df[column], errors="coerce")
        .fillna(0.0)
        .astype("float32")
    )


def _runner_flags(df):
    r1 = _numeric_or_zero(df, "runner_on_1b").gt(0)
    r2 = _numeric_or_zero(df, "runner_on_2b").gt(0)
    r3 = _numeric_or_zero(df, "runner_on_3b").gt(0)

    if not {"runner_on_1b", "runner_on_2b", "runner_on_3b"}.intersection(df.columns):
        # 개별 주자 플래그가 정말 없을 때만 num_runners_on으로 최소 fallback
        nr = _numeric_or_zero(df, "num_runners_on")
        any_runner = nr.gt(0)
        r1 = any_runner
        r2 = pd.Series(False, index=df.index)
        r3 = pd.Series(False, index=df.index)

    return r1, r2, r3


def _pitcher_team_score_diff(df):
    # 공식 pitcher-team 기준 score diff가 있으면 우선 사용
    if "score_diff_pitcher_team" in df.columns:
        return (
            pd.to_numeric(df["score_diff_pitcher_team"], errors="coerce")
            .astype("float32")
        )

    # raw score로 안전하게 재구성
    if {"run_top_before", "run_bot_before", "top_bottom"}.issubset(df.columns):
        top = pd.to_numeric(df["run_top_before"], errors="coerce")
        bot = pd.to_numeric(df["run_bot_before"], errors="coerce")

        # top 공격 시 투수는 홈팀, bottom 공격 시 투수는 원정팀
        diff = np.where(
            df["top_bottom"].eq("T"),
            bot - top,
            top - bot,
        )
        return pd.Series(diff, index=df.index, dtype="float32")

    return pd.Series(np.nan, index=df.index, dtype="float32")


def add_domain_static_features(df):
    df = df.copy()

    balls = pd.to_numeric(df["balls_before"], errors="coerce").fillna(-1).astype(int)
    strikes = pd.to_numeric(df["strikes_before"], errors="coerce").fillna(-1).astype(int)

    count_pairs = list(zip(balls.tolist(), strikes.tolist()))
    df["count_pressure"] = [
        (
            "pitcher_ahead"
            if pair in PITCHER_AHEAD_COUNTS
            else "batter_ahead"
            if pair in BATTER_AHEAD_COUNTS
            else "even"
        )
        for pair in count_pairs
    ]

    df["is_two_strike"] = strikes.eq(2).astype("int8")
    df["is_three_ball"] = balls.eq(3).astype("int8")
    df["is_full_count"] = (balls.eq(3) & strikes.eq(2)).astype("int8")
    df["is_0_2"] = (balls.eq(0) & strikes.eq(2)).astype("int8")
    df["is_3_0"] = (balls.eq(3) & strikes.eq(0)).astype("int8")

    # count × pitcher tendency: 과도한 interaction 확장은 피하고 3개만 사용
    df["two_strike_x_breaking"] = (
        df["is_two_strike"].astype("float32")
        * pd.to_numeric(df["asof_pitcher_breaking_rate"], errors="coerce").astype("float32")
    )
    df["three_ball_x_strike"] = (
        df["is_three_ball"].astype("float32")
        * pd.to_numeric(df["asof_pitcher_strike_rate"], errors="coerce").astype("float32")
    )
    df["three_ball_x_success"] = (
        df["is_three_ball"].astype("float32")
        * pd.to_numeric(df["asof_pitcher_success_rate"], errors="coerce").astype("float32")
    )

    r1, r2, r3 = _runner_flags(df)
    num_runners = (r1.astype("int8") + r2.astype("int8") + r3.astype("int8")).astype("int8")

    # 별도 이름을 사용하여 과거에 noise 제거했던 raw num_runners_on과 구분
    df["domain_num_runners_on"] = num_runners
    df["from_stretch"] = num_runners.gt(0).astype("int8")
    df["runner_1b_only"] = (r1 & ~r2 & ~r3).astype("int8")
    df["runner_scoring_position"] = (r2 | r3).astype("int8")
    df["runner_on_3b_less2out"] = (
        r3
        & pd.to_numeric(df["outs_before"], errors="coerce").fillna(3).lt(2)
    ).astype("int8")

    score_diff = _pitcher_team_score_diff(df)
    df["pitcher_team_score_diff_domain"] = score_diff
    df["abs_score_diff"] = score_diff.abs().astype("float32")
    df["is_blowout"] = df["abs_score_diff"].ge(6).fillna(False).astype("int8")

    inning = pd.to_numeric(df["inning"], errors="coerce")
    df["late_close"] = (
        inning.ge(7)
        & df["abs_score_diff"].le(3)
    ).fillna(False).astype("int8")

    df["same_side"] = (
        df["pitcher_hand"].astype(str)
        .eq(df["batter_hand"].astype(str))
        .astype("int8")
    )

    prev_cols = [
        "asof_pitcher_prev1_game_success_rate",
        "asof_pitcher_prev3_game_success_rate",
        "asof_pitcher_prev5_game_success_rate",
    ]
    prev1 = pd.to_numeric(df[prev_cols[0]], errors="coerce")
    prev3 = pd.to_numeric(df[prev_cols[1]], errors="coerce")
    prev5 = pd.to_numeric(df[prev_cols[2]], errors="coerce")

    df["prev1_minus_prev5_success"] = (prev1 - prev5).astype("float32")
    df["prev3_minus_prev5_success"] = (prev3 - prev5).astype("float32")

    recent_matrix = pd.concat([prev1, prev3, prev5], axis=1)
    df["recent_success_range"] = (
        recent_matrix.max(axis=1, skipna=True)
        - recent_matrix.min(axis=1, skipna=True)
    ).astype("float32")

    df["log1p_asof_pitcher_n"] = np.log1p(
        pd.to_numeric(df["asof_pitcher_n"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    ).astype("float32")

    return df

## 5. Strict-season history reconstruction

현재 투구 결과를 다음 row의 ASOF 누적 통계 변화로 복원하되,
복원된 flag는 **동일 시즌 현재 row의 모델 입력으로 직접 사용하지 않습니다.**

용도는 오직:

`과거 시즌 결과 → 이후 시즌의 Flat / Current Season history`

입니다.

In [7]:
# ============================================================
# 5. Flat-history definitions + chronological flag reconstruction
# ============================================================

FLAG_COLS = [
    "flag_success",
    "flag_middle",
    "flag_reverse",
    "flag_ball",
    "flag_strike",
    "flag_fastball",
    "flag_breaking",
    "flag_offspeed",
]

FLAT_GROUPS = {
    "league_count": ["count_state"],
    "league_count_hand": ["count_state", "batter_hand"],
    "pitcher": ["pitcher_id"],
    "pitcher_hand": ["pitcher_id", "batter_hand"],
    "pitcher_count": ["pitcher_id", "count_state"],
    "pitcher_count_hand": ["pitcher_id", "count_state", "batter_hand"],
    "pitcher_baseout": ["pitcher_id", "base_out_state"],
    "pitcher_count_baseout": ["pitcher_id", "count_state", "base_out_state"],
    "handmatch_count": ["hand_matchup", "count_state"],
}

ASOF_FLAG_MAP = {
    "flag_success": "asof_pitcher_success_rate",
    "flag_middle": "asof_pitcher_middle_rate",
    "flag_reverse": "asof_pitcher_reverse_rate",
    "flag_ball": "asof_pitcher_ball_rate",
    "flag_strike": "asof_pitcher_strike_rate",
    "flag_fastball": "asof_pitcher_fastball_rate",
    "flag_breaking": "asof_pitcher_breaking_rate",
    "flag_offspeed": "asof_pitcher_offspeed_rate",
}

PITCHER_CONTEXTS = [
    "pitcher_count",
    "pitcher_hand",
    "pitcher_count_hand",
    "pitcher_baseout",
]

CURRENT_SEASON_FEATURES = [
    "current_season_pitcher_n",
    "current_season_pitcher_success_rate",
    "current_season_pitcher_success_shrunk",
    "current_season_pitcher_reliability",
    "current_season_minus_career_success",
]


def reconstruct_pitch_flags(data, target_column=TARGET):
    data = data.copy()
    work = data.copy()

    work["_original_position"] = np.arange(len(work), dtype=np.int64)

    work = (
        work.sort_values(
            [
                "pitcher_id",
                "season",
                "asof_pitcher_n",
                "_original_position",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    group_keys = ["pitcher_id", "season"]

    n_diff = (
        work.groupby(group_keys, sort=False)["asof_pitcher_n"]
        .diff()
    )
    if n_diff.dropna().lt(0).any():
        raise AssertionError(
            "Chronological ordering failed: asof_pitcher_n decreased "
            "inside a pitcher-season group."
        )

    pitcher_group = work.groupby(group_keys, sort=False)

    def reconstruct_flag(rate_column, n_column):
        n_now = pd.to_numeric(work[n_column], errors="coerce").astype("float64")
        n_next = (
            pitcher_group[n_column]
            .shift(-1)
            .astype("float64")
        )

        rate_now = (
            pd.to_numeric(work[rate_column], errors="coerce")
            .fillna(0.0)
            .astype("float64")
        )
        rate_next = (
            pitcher_group[rate_column]
            .shift(-1)
            .astype("float64")
        )

        raw_flag = n_next * rate_next - n_now * rate_now
        valid = n_next.eq(n_now + 1) & rate_next.notna()

        reconstructed = pd.Series(
            np.nan,
            index=work.index,
            dtype="float32",
        )
        reconstructed.loc[valid] = (
            np.rint(raw_flag.loc[valid])
            .clip(0, 1)
            .astype("float32")
        )
        return reconstructed

    work["flag_success"] = pd.to_numeric(
        work[target_column], errors="raise"
    ).astype("float32")

    work["flag_middle"] = reconstruct_flag(
        "asof_pitcher_middle_rate", "asof_pitcher_n"
    )
    work["flag_reverse"] = reconstruct_flag(
        "asof_pitcher_reverse_rate", "asof_pitcher_n"
    )
    work["flag_ball"] = reconstruct_flag(
        "asof_pitcher_ball_rate", "asof_pitcher_n"
    )
    work["flag_strike"] = reconstruct_flag(
        "asof_pitcher_strike_rate", "asof_pitcher_n"
    )
    work["flag_fastball"] = reconstruct_flag(
        "asof_pitcher_fastball_rate", "asof_pitcher_pitchmix_n"
    )
    work["flag_breaking"] = reconstruct_flag(
        "asof_pitcher_breaking_rate", "asof_pitcher_pitchmix_n"
    )
    work["flag_offspeed"] = reconstruct_flag(
        "asof_pitcher_offspeed_rate", "asof_pitcher_pitchmix_n"
    )

    work = work.sort_values("_original_position").reset_index(drop=True)

    for flag in FLAG_COLS:
        data[flag] = work[flag].to_numpy(copy=False)

    rates = {
        flag: float(data[flag].notna().mean())
        for flag in FLAG_COLS
        if flag != "flag_success"
    }
    print("Flag reconstruction valid-rate:")
    print(pd.Series(rates).round(4))

    return data

In [8]:
# ============================================================
# 6. Strict-season Flat History
# ============================================================

def calculate_flat_priors(history):
    priors = {}
    for flag in FLAG_COLS:
        values = history[flag].dropna()
        priors[flag] = float(values.mean()) if len(values) else np.nan
    return priors


def make_flat_history_features(target_data, history_data, flat_alpha=FLAT_ALPHA):
    target_order = np.arange(len(target_data), dtype=np.int32)
    priors = calculate_flat_priors(history_data)

    feature_parts = []

    for group_name, group_columns in FLAT_GROUPS.items():
        if len(history_data):
            aggregation = {}
            for flag in FLAG_COLS:
                aggregation[f"__{flag}__sum"] = (flag, "sum")
                aggregation[f"__{flag}__count"] = (flag, "count")

            history_summary = (
                history_data
                .groupby(
                    group_columns,
                    dropna=False,
                    observed=False,
                )
                .agg(**aggregation)
                .reset_index()
            )

            merged = target_data[group_columns].copy()
            merged["_flat_order"] = target_order
            merged = merged.merge(
                history_summary,
                on=group_columns,
                how="left",
                sort=False,
            )
        else:
            merged = target_data[group_columns].copy()
            merged["_flat_order"] = target_order
            for flag in FLAG_COLS:
                merged[f"__{flag}__sum"] = 0.0
                merged[f"__{flag}__count"] = 0

        feature_data = {}

        for flag in FLAG_COLS:
            n = (
                merged[f"__{flag}__count"]
                .fillna(0)
                .astype("int32")
            )
            flag_sum = (
                merged[f"__{flag}__sum"]
                .fillna(0.0)
                .astype("float64")
            )

            prior = priors[flag]

            if np.isfinite(prior):
                rate = (
                    (flag_sum + float(flat_alpha) * prior)
                    / (n.astype("float64") + float(flat_alpha))
                )
            else:
                rate = pd.Series(
                    np.nan,
                    index=merged.index,
                    dtype="float64",
                )

            feature_data[f"flat_{group_name}_{flag}_rate"] = (
                rate.astype("float32").to_numpy()
            )
            feature_data[f"flat_{group_name}_{flag}_n"] = (
                n.to_numpy(dtype=np.int32, copy=False)
            )

        part = pd.DataFrame(
            feature_data,
            index=merged["_flat_order"].to_numpy(),
        )
        feature_parts.append(part)

    result = pd.concat(feature_parts, axis=1).sort_index()
    result.index = target_data.index
    return result

In [9]:
# ============================================================
# 7. Current Season Decomposition
# ============================================================

def make_current_season_features(
    target_data,
    history_data,
    current_season_alpha=CURRENT_SEASON_ALPHA,
):
    if len(history_data):
        prior_summary = (
            history_data
            .groupby(
                "pitcher_id",
                dropna=False,
                observed=True,
            )
            .agg(
                prior_pitcher_success_sum=("flag_success", "sum"),
                prior_pitcher_n=("flag_success", "count"),
            )
            .reset_index()
        )
        league_prior = float(history_data["flag_success"].mean())
    else:
        prior_summary = pd.DataFrame(
            columns=[
                "pitcher_id",
                "prior_pitcher_success_sum",
                "prior_pitcher_n",
            ]
        )
        # 첫 시즌은 rolling validation 대상이 아니므로 중립 prior 사용
        league_prior = 0.5

    target_order = np.arange(len(target_data), dtype=np.int32)
    merged = target_data[
        ["pitcher_id", "asof_pitcher_n", "asof_pitcher_success_rate"]
    ].copy()
    merged["_current_order"] = target_order

    merged = merged.merge(
        prior_summary,
        on="pitcher_id",
        how="left",
        sort=False,
    )

    prior_n = (
        pd.to_numeric(merged["prior_pitcher_n"], errors="coerce")
        .fillna(0)
        .astype("float32")
    )
    prior_sum = (
        pd.to_numeric(merged["prior_pitcher_success_sum"], errors="coerce")
        .fillna(0.0)
        .astype("float32")
    )

    asof_n = (
        pd.to_numeric(merged["asof_pitcher_n"], errors="coerce")
        .fillna(0)
        .astype("float32")
    )
    asof_rate = (
        pd.to_numeric(merged["asof_pitcher_success_rate"], errors="coerce")
        .astype("float32")
    )

    asof_success_sum = (asof_n * asof_rate.fillna(0.0)).astype("float32")

    current_n = np.maximum(
        0.0,
        asof_n.to_numpy(copy=False) - prior_n.to_numpy(copy=False),
    ).astype("float32")

    current_sum = np.maximum(
        0.0,
        asof_success_sum.to_numpy(copy=False) - prior_sum.to_numpy(copy=False),
    ).astype("float32")

    current_sum = np.minimum(current_sum, current_n).astype("float32")

    current_rate = np.divide(
        current_sum,
        current_n,
        out=np.full(len(target_data), np.nan, dtype=np.float32),
        where=current_n >= 1.0,
    ).astype("float32")

    alpha = float(current_season_alpha)
    current_shrunk = (
        (current_sum + alpha * league_prior)
        / (current_n + alpha)
    ).astype("float32")

    reliability = (
        current_n / (current_n + alpha)
    ).astype("float32")

    minus_career = (
        current_shrunk
        - asof_rate.to_numpy(dtype=np.float32, copy=False)
    ).astype("float32")

    result = pd.DataFrame(
        {
            "current_season_pitcher_n": current_n,
            "current_season_pitcher_success_rate": current_rate,
            "current_season_pitcher_success_shrunk": current_shrunk,
            "current_season_pitcher_reliability": reliability,
            "current_season_minus_career_success": minus_career,
        },
        index=merged["_current_order"].to_numpy(),
    ).sort_index()

    result.index = target_data.index
    return result[CURRENT_SEASON_FEATURES]

In [10]:
# ============================================================
# 8. Drift / Safe Context / HLogit
# ============================================================

def probability_to_logit(values, eps=LOGIT_EPS):
    values = np.asarray(values, dtype=np.float64)
    values = np.clip(values, eps, 1.0 - eps)
    return np.log(values / (1.0 - values))


def add_pitcher_context_features(df):
    df = df.copy()
    feature_data = {}

    alpha = float(CONTEXT_EFFECT_ALPHA)

    for flag, asof_col in ASOF_FLAG_MAP.items():
        current_rate = pd.to_numeric(df[asof_col], errors="coerce").astype("float32")
        overall_rate = df[f"flat_pitcher_{flag}_rate"].astype("float32")

        feature_data[f"pitcher_drift_{flag}"] = (
            current_rate - overall_rate
        ).astype("float32").to_numpy(copy=False)

        for context in PITCHER_CONTEXTS:
            context_rate = df[f"flat_{context}_{flag}_rate"].astype("float32")
            context_n = df[f"flat_{context}_{flag}_n"].astype("float32")

            reliability = (
                context_n / (context_n + alpha)
            ).astype("float32")

            effect = (
                (context_rate - overall_rate) * reliability
            ).astype("float32")

            feature_data[
                f"pitcher_context_effect_{context}_{flag}"
            ] = effect.to_numpy(copy=False)

            feature_data[
                f"pitcher_safe_context_{context}_{flag}"
            ] = np.clip(
                current_rate.to_numpy(dtype=np.float32, copy=False)
                + effect.to_numpy(dtype=np.float32, copy=False),
                0.0,
                1.0,
            ).astype("float32")

    # HLogit
    for flag in FLAG_COLS:
        overall = df[f"flat_pitcher_{flag}_rate"].to_numpy(dtype=np.float32)
        count = df[f"flat_pitcher_count_{flag}_rate"].to_numpy(dtype=np.float32)
        hand = df[f"flat_pitcher_hand_{flag}_rate"].to_numpy(dtype=np.float32)
        count_hand = df[f"flat_pitcher_count_hand_{flag}_rate"].to_numpy(dtype=np.float32)
        baseout = df[f"flat_pitcher_baseout_{flag}_rate"].to_numpy(dtype=np.float32)

        overall_logit = probability_to_logit(overall)
        count_logit = probability_to_logit(count)
        hand_logit = probability_to_logit(hand)
        count_hand_logit = probability_to_logit(count_hand)
        baseout_logit = probability_to_logit(baseout)

        count_n = df[f"flat_pitcher_count_{flag}_n"].to_numpy(dtype=np.float32)
        hand_n = df[f"flat_pitcher_hand_{flag}_n"].to_numpy(dtype=np.float32)
        count_hand_n = df[f"flat_pitcher_count_hand_{flag}_n"].to_numpy(dtype=np.float32)
        baseout_n = df[f"flat_pitcher_baseout_{flag}_n"].to_numpy(dtype=np.float32)

        count_rel = (count_n / (count_n + alpha)).astype("float32")
        hand_rel = (hand_n / (hand_n + alpha)).astype("float32")
        count_hand_rel = (count_hand_n / (count_hand_n + alpha)).astype("float32")
        baseout_rel = (baseout_n / (baseout_n + alpha)).astype("float32")

        feature_data[f"pitcher_hlogit_count_{flag}"] = (
            (count_logit - overall_logit) * count_rel
        ).astype("float32")

        feature_data[f"pitcher_hlogit_hand_{flag}"] = (
            (hand_logit - overall_logit) * hand_rel
        ).astype("float32")

        interaction_rel = np.minimum(
            np.minimum(count_rel, hand_rel),
            count_hand_rel,
        ).astype("float32")

        feature_data[
            f"pitcher_hlogit_count_hand_interaction_{flag}"
        ] = (
            (
                count_hand_logit
                - count_logit
                - hand_logit
                + overall_logit
            )
            * interaction_rel
        ).astype("float32")

        feature_data[f"pitcher_hlogit_baseout_{flag}"] = (
            (baseout_logit - overall_logit) * baseout_rel
        ).astype("float32")

    return pd.concat(
        [df, pd.DataFrame(feature_data, index=df.index)],
        axis=1,
    )

## 9. Pitcher role — strict historical lookup

역할 라벨은 target을 사용하지 않고 `inning` 분포만 사용합니다.

- `starter`: 과거 투구가 초반 이닝에 충분히 집중되고 평균 이닝도 이른 편
- `closer_like`: 8회 이후 투구 비중이 높은 투수
- `reliever`: 그 외
- `unknown`: 과거 표본 부족

중요: 2024 row의 역할은 2019~2023만 보고 만듭니다.

In [11]:
# ============================================================
# 9. Strict-history pitcher role
# ============================================================

ROLE_FEATURES = [
    "role_history_n",
    "role_history_n_log1p",
    "role_mean_inning",
    "role_early_share",
    "role_late_share",
    "pitcher_role",
    "role_inning_state",
    "role_starter_x_inning",
    "role_reliever_x_inning",
]


def make_role_lookup(history):
    if len(history) == 0:
        return pd.DataFrame(
            columns=[
                "pitcher_id",
                "role_history_n",
                "role_mean_inning",
                "role_early_share",
                "role_late_share",
                "pitcher_role",
            ]
        )

    work = history[["pitcher_id", "inning"]].copy()
    work["_inning_num"] = pd.to_numeric(work["inning"], errors="coerce")
    work["_early"] = work["_inning_num"].le(3).astype("float32")
    work["_late"] = work["_inning_num"].ge(8).astype("float32")

    lookup = (
        work.groupby(
            "pitcher_id",
            dropna=False,
            observed=True,
        )
        .agg(
            role_history_n=("_inning_num", "count"),
            role_mean_inning=("_inning_num", "mean"),
            role_early_share=("_early", "mean"),
            role_late_share=("_late", "mean"),
        )
        .reset_index()
    )

    n = lookup["role_history_n"]
    starter = (
        n.ge(ROLE_MIN_PITCHES)
        & lookup["role_early_share"].ge(ROLE_STARTER_EARLY_SHARE)
        & lookup["role_mean_inning"].le(ROLE_STARTER_MEAN_INNING_MAX)
    )
    closer = (
        n.ge(ROLE_MIN_PITCHES)
        & lookup["role_late_share"].ge(ROLE_CLOSER_LATE_SHARE)
    )

    lookup["pitcher_role"] = np.select(
        [
            n.lt(ROLE_MIN_PITCHES),
            closer,
            starter,
        ],
        [
            "unknown",
            "closer_like",
            "starter",
        ],
        default="reliever",
    )

    return lookup


def apply_role_lookup(target_data, role_lookup):
    target = target_data[["pitcher_id", "inning"]].copy()
    target["_role_order"] = np.arange(len(target), dtype=np.int32)

    merged = target.merge(
        role_lookup,
        on="pitcher_id",
        how="left",
        sort=False,
    ).sort_values("_role_order")

    merged["role_history_n"] = (
        pd.to_numeric(merged["role_history_n"], errors="coerce")
        .fillna(0)
        .astype("float32")
    )
    merged["role_history_n_log1p"] = np.log1p(
        merged["role_history_n"].clip(lower=0)
    ).astype("float32")

    for c in ["role_mean_inning", "role_early_share", "role_late_share"]:
        merged[c] = pd.to_numeric(merged[c], errors="coerce").astype("float32")

    merged["pitcher_role"] = (
        merged["pitcher_role"]
        .fillna("unknown")
        .astype(str)
    )

    inning_num = pd.to_numeric(merged["inning"], errors="coerce").fillna(-1)

    merged["role_inning_state"] = (
        merged["pitcher_role"]
        + "_inn"
        + inning_num.astype(int).astype(str)
    )

    merged["role_starter_x_inning"] = (
        merged["pitcher_role"].eq("starter").astype("float32")
        * inning_num.astype("float32")
    )
    merged["role_reliever_x_inning"] = (
        merged["pitcher_role"].isin(["reliever", "closer_like"]).astype("float32")
        * inning_num.astype("float32")
    )

    result = merged[ROLE_FEATURES].copy()
    result.index = target_data.index
    return result

## 10. 전체 Train Feature Matrix 생성

핵심은 모든 시즌에 대해:

`target = season == S`  
`history = season < S`

로 만든다는 점입니다.

따라서 rolling CV를 나중에 적용해도, validation row의 history feature가 자기 시즌 target을 보지 않습니다.

In [12]:
# ============================================================
# 10. Build strict-season training features
# ============================================================

def build_strict_training_features(train_input):
    base = add_base_features(train_input)
    base = add_domain_static_features(base)
    base = reconstruct_pitch_flags(base, TARGET)

    history_cols = [
        "season",
        "pitcher_id",
        "count_state",
        "batter_hand",
        "base_out_state",
        "hand_matchup",
        "inning",
        *FLAG_COLS,
    ]
    history_compact = base[history_cols].copy()

    flat_parts = []
    current_parts = []
    role_parts = []

    seasons = sorted(base["season"].dropna().unique().tolist())

    for target_season in seasons:
        target_rows = base.loc[base["season"].eq(target_season)]
        history_rows = history_compact.loc[
            history_compact["season"].lt(target_season)
        ]

        # Hard safety assertion
        if len(history_rows):
            assert history_rows["season"].max() < target_season

        flat_part = make_flat_history_features(
            target_rows,
            history_rows,
            FLAT_ALPHA,
        )
        current_part = make_current_season_features(
            target_rows,
            history_rows,
            CURRENT_SEASON_ALPHA,
        )
        role_lookup = make_role_lookup(history_rows)
        role_part = apply_role_lookup(target_rows, role_lookup)

        flat_parts.append(flat_part)
        current_parts.append(current_part)
        role_parts.append(role_part)

        print(
            f"Season {target_season} | "
            f"target={len(target_rows):,} | "
            f"history={len(history_rows):,} | "
            f"role_pitchers={len(role_lookup):,}"
        )

        del target_rows, history_rows, flat_part, current_part, role_lookup, role_part
        gc.collect()

    flat_all = pd.concat(flat_parts, axis=0).reindex(base.index)
    current_all = pd.concat(current_parts, axis=0).reindex(base.index)
    role_all = pd.concat(role_parts, axis=0).reindex(base.index)

    model_df = pd.concat(
        [
            base.drop(columns=FLAG_COLS),
            flat_all,
            current_all,
            role_all,
        ],
        axis=1,
    )

    model_df = add_pitcher_context_features(model_df)

    # first-season missing: target outcome을 쓰지 않는 식별 정보
    first_season = (
        train_input.groupby("pitcher_id", observed=True)["season"]
        .min()
    )
    model_df["preseason_pitcher_missing"] = (
        model_df["pitcher_id"]
        .map(first_season)
        .isna()
        | model_df["pitcher_id"].map(first_season).ge(model_df["season"])
    ).astype("int8")

    return model_df, history_compact, first_season


train, history_compact, pitcher_first_season = build_strict_training_features(
    train_raw
)

print("Final engineered train shape:", train.shape)

Flag reconstruction valid-rate:
flag_middle      0.9985
flag_reverse     0.9985
flag_ball        0.9985
flag_strike      0.9985
flag_fastball    0.9985
flag_breaking    0.9985
flag_offspeed    0.9985
dtype: float64
Season 2019 | target=237,413 | history=0 | role_pitchers=0
Season 2020 | target=244,087 | history=237,413 | role_pitchers=355
Season 2021 | target=247,088 | history=481,500 | role_pitchers=465
Season 2022 | target=247,472 | history=728,588 | role_pitchers=560
Season 2023 | target=245,525 | history=976,060 | role_pitchers=648
Season 2024 | target=253,507 | history=1,221,585 | role_pitchers=711
Final engineered train shape: (1475092, 349)


In [13]:
# ============================================================
# 11. Define baseline + Domain v1 feature registry
# ============================================================

# 최신 minus_batter_flat 방향에서 제거했던 noise/raw identity 피처를 유지
RAW_EXCLUDE = {
    ID,
    TARGET,
    "pitcher_id",
    "batter_id",
    "asof_batter_middle_rate",
    "batter_history_missing",
    "runner_on_1b",
    "runner_on_2b",
    "runner_on_3b",
    "num_runners_on",
    *FLAG_COLS,
}

if test_raw is not None:
    raw_source_columns = [
        c for c in train_raw.columns
        if c in test_raw.columns
    ]
else:
    raw_source_columns = list(train_raw.columns)

BASE_RAW_FEATURES = [
    c for c in raw_source_columns
    if c not in RAW_EXCLUDE
]

BASE_ENGINEERED_FEATURES = [
    "count_state",
    "base_out_state",
    "hand_matchup",
    "pitcher_is_home",
    "pitcher_team_win_expectancy",
    "pitcher_history_missing",
    "previous_game_history_missing",
    "pitcher_success_prev1_delta",
    "pitcher_success_prev3_delta",
    "pitcher_success_prev5_delta",
    "pitcher_success_count500",
    "pitcher_reverse_count500",
    "pitcher_ball_count500",
    "pitcher_strike_count500",
]

FLAT_FEATURES = [
    c for c in train.columns
    if c.startswith("flat_")
]

CONTEXT_FEATURES = [
    c for c in train.columns
    if (
        c.startswith("pitcher_drift_")
        or c.startswith("pitcher_context_effect_")
        or c.startswith("pitcher_safe_context_")
        or c.startswith("pitcher_hlogit_")
    )
]

EXISTING_HISTORY_FEATURES = [
    *FLAT_FEATURES,
    *CURRENT_SEASON_FEATURES,
    *CONTEXT_FEATURES,
    "preseason_pitcher_missing",
]

# -----------------------
# Domain v1 groups
# -----------------------
DOMAIN_GROUPS = {
    "count_pressure": [
        "count_pressure",
        "is_two_strike",
        "is_three_ball",
        "is_full_count",
        "is_0_2",
        "is_3_0",
        "two_strike_x_breaking",
        "three_ball_x_strike",
        "three_ball_x_success",
    ],
    "runner_stretch": [
        "domain_num_runners_on",
        "from_stretch",
        "runner_1b_only",
        "runner_scoring_position",
        "runner_on_3b_less2out",
    ],
    "game_pressure": [
        "pitcher_team_score_diff_domain",
        "abs_score_diff",
        "is_blowout",
        "late_close",
    ],
    "matchup": [
        "same_side",
    ],
    "recent_trend": [
        "prev1_minus_prev5_success",
        "prev3_minus_prev5_success",
        "recent_success_range",
    ],
    "sample_size": [
        "log1p_asof_pitcher_n",
    ],
    "role": list(ROLE_FEATURES),
}

DOMAIN_FEATURES = [
    f
    for group_features in DOMAIN_GROUPS.values()
    for f in group_features
]

def unique_existing(columns):
    seen = set()
    out = []
    for c in columns:
        if c in train.columns and c not in seen:
            out.append(c)
            seen.add(c)
    return out

BASELINE_FEATURES = unique_existing(
    BASE_RAW_FEATURES
    + BASE_ENGINEERED_FEATURES
    + EXISTING_HISTORY_FEATURES
)

DOMAIN_FEATURES = unique_existing(DOMAIN_FEATURES)

FEATURES_DOMAIN_FULL = unique_existing(
    BASELINE_FEATURES + DOMAIN_FEATURES
)

print("Base raw              :", len(BASE_RAW_FEATURES))
print("Base engineered       :", len(BASE_ENGINEERED_FEATURES))
print("Flat history          :", len(FLAT_FEATURES))
print("Context/HLogit        :", len(CONTEXT_FEATURES))
print("Baseline total        :", len(BASELINE_FEATURES))
print("Domain v1 added       :", len(DOMAIN_FEATURES))
print("Domain v1 full total  :", len(FEATURES_DOMAIN_FULL))
print()
for group_name, cols in DOMAIN_GROUPS.items():
    kept = [c for c in cols if c in DOMAIN_FEATURES]
    print(f"{group_name:16s}: {len(kept):2d} | {kept}")

Base raw              : 40
Base engineered       : 14
Flat history          : 144
Context/HLogit        : 104
Baseline total        : 308
Domain v1 added       : 32
Domain v1 full total  : 340

count_pressure  :  9 | ['count_pressure', 'is_two_strike', 'is_three_ball', 'is_full_count', 'is_0_2', 'is_3_0', 'two_strike_x_breaking', 'three_ball_x_strike', 'three_ball_x_success']
runner_stretch  :  5 | ['domain_num_runners_on', 'from_stretch', 'runner_1b_only', 'runner_scoring_position', 'runner_on_3b_less2out']
game_pressure   :  4 | ['pitcher_team_score_diff_domain', 'abs_score_diff', 'is_blowout', 'late_close']
matchup         :  1 | ['same_side']
recent_trend    :  3 | ['prev1_minus_prev5_success', 'prev3_minus_prev5_success', 'recent_success_range']
sample_size     :  1 | ['log1p_asof_pitcher_n']
role            :  9 | ['role_history_n', 'role_history_n_log1p', 'role_mean_inning', 'role_early_share', 'role_late_share', 'pitcher_role', 'role_inning_state', 'role_starter_x_inning', 'rol

In [14]:
# ============================================================
# 12. Categorical columns + CatBoost-safe missing handling
# ============================================================

# 기존 학습 코드에서 categorical로 취급했던 raw columns를 우선 보존
LEGACY_KNOWN_CATEGORICAL = [
    "game_dayofweek",
    "top_bottom",
    "game_type",
    "base_state",
    "pitcher_hand",
    "batter_hand",
    "pitcher_team_id",
    "batter_team_id",
    "count_state",
    "base_out_state",
    "hand_matchup",
]

# 그 외 raw object/string columns도 자동 탐지
RAW_CATEGORICAL = [
    c for c in BASE_RAW_FEATURES
    if (
        pd.api.types.is_object_dtype(train[c])
        or pd.api.types.is_string_dtype(train[c])
        or isinstance(train[c].dtype, pd.CategoricalDtype)
    )
]

DOMAIN_CATEGORICAL = [
    "count_pressure",
    "pitcher_role",
    "role_inning_state",
]

CAT_COLS = unique_existing(
    LEGACY_KNOWN_CATEGORICAL
    + RAW_CATEGORICAL
    + DOMAIN_CATEGORICAL
)

# CatBoost categorical feature는 NaN float를 허용하지 않으므로 문자열로 통일
for c in CAT_COLS:
    train[c] = (
        train[c]
        .astype("object")
        .where(train[c].notna(), "__MISSING__")
        .astype(str)
    )

# 숫자 컬럼에 +/-inf가 있으면 NaN으로 변경
numeric_features = [
    c for c in FEATURES_DOMAIN_FULL
    if c not in CAT_COLS
]
for c in numeric_features:
    if pd.api.types.is_numeric_dtype(train[c]):
        arr = pd.to_numeric(train[c], errors="coerce")
        train[c] = arr.replace([np.inf, -np.inf], np.nan)

print("Categorical feature count:", len(CAT_COLS))
print(CAT_COLS)

Categorical feature count: 14
['game_dayofweek', 'top_bottom', 'game_type', 'base_state', 'pitcher_hand', 'batter_hand', 'pitcher_team_id', 'batter_team_id', 'count_state', 'base_out_state', 'hand_matchup', 'count_pressure', 'pitcher_role', 'role_inning_state']


## 13. Rolling CV

최근 3개 시즌을 validation으로 사용합니다.

각 fold:

- train: `season < valid_season`
- valid: `season == valid_season`

평가:

- Brier Score
- Competition-style normalized score
- fold별 best iteration
- prediction mean/std

In [15]:
# ============================================================
# 13. Rolling CV setup
# ============================================================

all_seasons = sorted(
    train["season"].dropna().unique().tolist()
)

if len(all_seasons) < 2:
    raise ValueError("Rolling CV를 위해 최소 2개 시즌이 필요합니다.")

eligible_valid_seasons = all_seasons[1:]
ROLLING_VALID_SEASONS = eligible_valid_seasons[
    -min(N_ROLLING_FOLDS, len(eligible_valid_seasons)):
]

print("All seasons:", all_seasons)
print("Rolling validation seasons:", ROLLING_VALID_SEASONS)


def brier_score_np(y_true, probability):
    y_true = np.asarray(y_true, dtype=np.float64)
    probability = np.asarray(probability, dtype=np.float64)
    return float(np.mean((probability - y_true) ** 2))


def competition_score_from_brier(y_true, probability):
    y_true = np.asarray(y_true, dtype=np.float64)
    brier = brier_score_np(y_true, probability)

    prevalence = float(y_true.mean())
    baseline_brier = prevalence * (1.0 - prevalence)

    if baseline_brier <= 0:
        return brier, np.nan

    score = max(
        0.0,
        100000.0 * (1.0 - brier / baseline_brier),
    )
    return brier, score


BASE_CATBOOST_PARAMS = {
    "loss_function": "Logloss",
    "eval_metric": "BrierScore",
    "iterations": 900,
    "learning_rate": 0.04,
    "depth": 6,
    "l2_leaf_reg": 7.0,
    "random_strength": 1.0,
    "max_ctr_complexity": 1,
    "border_count": 64,
    "random_seed": 42,
    "task_type": "CPU",
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": False,
}

All seasons: [2019, 2020, 2021, 2022, 2023, 2024]
Rolling validation seasons: [2022, 2023, 2024]


In [17]:
# ============================================================
# 14. Rolling OOF runner
# ============================================================

def run_rolling_oof(
    feature_list,
    experiment_name,
    valid_seasons=None,
    params_override=None,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
):
    if valid_seasons is None:
        valid_seasons = ROLLING_VALID_SEASONS

    feature_list = unique_existing(feature_list)

    missing = [c for c in feature_list if c not in train.columns]
    if missing:
        raise KeyError(f"{experiment_name}: missing features={missing[:20]}")

    cat_cols = [
        c for c in CAT_COLS
        if c in feature_list
    ]

    params = dict(BASE_CATBOOST_PARAMS)
    if params_override:
        params.update(params_override)

    oof_probability = pd.Series(
        np.nan,
        index=train.index,
        dtype="float64",
    )

    fold_rows = []
    trees_used = []
    start_time = time.time()

    for valid_season in valid_seasons:
        train_mask = train["season"].lt(valid_season)
        valid_mask = train["season"].eq(valid_season)

        if not train_mask.any() or not valid_mask.any():
            raise ValueError(f"Invalid fold: {valid_season}")

        assert train.loc[train_mask, "season"].max() < valid_season

        model = CatBoostClassifier(**params)

        model.fit(
            train.loc[train_mask, feature_list],
            train.loc[train_mask, TARGET],
            cat_features=cat_cols,
            eval_set=(
                train.loc[valid_mask, feature_list],
                train.loc[valid_mask, TARGET],
            ),
            use_best_model=True,
            early_stopping_rounds=early_stopping_rounds,
            verbose=False,
        )

        pred = model.predict_proba(
            train.loc[valid_mask, feature_list]
        )[:, 1]

        oof_probability.loc[valid_mask] = pred

        fold_brier, fold_score = competition_score_from_brier(
            train.loc[valid_mask, TARGET].to_numpy(),
            pred,
        )

        best_idx = int(model.get_best_iteration())
        fold_trees = (
            best_idx + 1
            if best_idx >= 0
            else int(params["iterations"])
        )
        trees_used.append(fold_trees)

        fold_rows.append(
            {
                "experiment": experiment_name,
                "valid_season": valid_season,
                "n_train": int(train_mask.sum()),
                "n_valid": int(valid_mask.sum()),
                "brier": fold_brier,
                "score": fold_score,
                "trees_used": fold_trees,
                "pred_mean": float(np.mean(pred)),
                "pred_std": float(np.std(pred)),
            }
        )

        print(
            f"[{experiment_name}] valid={valid_season} | "
            f"Brier={fold_brier:.10f} | "
            f"trees={fold_trees} | "
            f"pred_std={np.std(pred):.6f}"
        )

        del model, pred
        gc.collect()

    valid_oof_mask = train["season"].isin(valid_seasons)
    y_oof = train.loc[valid_oof_mask, TARGET].to_numpy()
    p_oof = oof_probability.loc[valid_oof_mask].to_numpy()

    if np.isnan(p_oof).any():
        raise RuntimeError(f"{experiment_name}: OOF prediction has NaN.")

    oof_brier, oof_score = competition_score_from_brier(
        y_oof,
        p_oof,
    )

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "experiment": experiment_name,
        "oof_brier": float(oof_brier),
        "oof_score": float(oof_score),
        "mean_fold_brier": float(fold_df["brier"].mean()),
        "std_fold_brier": float(fold_df["brier"].std(ddof=0)),
        "worst_fold_brier": float(fold_df["brier"].max()),
        "median_trees_used": int(np.median(trees_used)),
        "n_features": int(len(feature_list)),
        "elapsed_sec": float(time.time() - start_time),
    }

    return {
        "summary": summary,
        "folds": fold_df,
        "oof_probability": oof_probability,
        "features": list(feature_list),
    }

## 15. Domain v1 Full vs 기존 Baseline

먼저 두 개를 비교합니다.

- `baseline_no_domain`: 기존 전처리만
- `domain_full`: 기존 전처리 + Domain v1 전체

이 결과가 Domain v1 전체의 순효과를 보여줍니다.

In [18]:
# ============================================================
# 15. Baseline and Domain-full rolling CV
# ============================================================

RESULT_REGISTRY = {}

baseline_result = run_rolling_oof(
    BASELINE_FEATURES,
    experiment_name="baseline_no_domain",
)
RESULT_REGISTRY["baseline_no_domain"] = baseline_result

domain_full_result = run_rolling_oof(
    FEATURES_DOMAIN_FULL,
    experiment_name="domain_full",
)
RESULT_REGISTRY["domain_full"] = domain_full_result

baseline_compare = pd.DataFrame(
    [
        baseline_result["summary"],
        domain_full_result["summary"],
    ]
).sort_values("oof_brier")

baseline_brier = baseline_result["summary"]["oof_brier"]
baseline_compare["delta_brier_vs_baseline"] = (
    baseline_compare["oof_brier"] - baseline_brier
)

display(baseline_compare)

[baseline_no_domain] valid=2022 | Brier=0.2430209494 | trees=479 | pred_std=0.075094
[baseline_no_domain] valid=2023 | Brier=0.2499332573 | trees=4 | pred_std=0.009767
[baseline_no_domain] valid=2024 | Brier=0.2475855404 | trees=472 | pred_std=0.047385
[domain_full] valid=2022 | Brier=0.2430445766 | trees=418 | pred_std=0.075075
[domain_full] valid=2023 | Brier=0.2499414456 | trees=4 | pred_std=0.010096
[domain_full] valid=2024 | Brier=0.2476131965 | trees=436 | pred_std=0.047144


,experiment,oof_brier,oof_score,mean_fold_brier,std_fold_brier,worst_fold_brier,median_trees_used,n_features,elapsed_sec,delta_brier_vs_baseline
0,baseline_no_domain,0.246845,1252.888808,0.246847,0.002870,0.249933,472,308,543.399646,0.00000
1,domain_full,0.246864,1244.921045,0.246866,0.002865,0.249941,418,340,625.306375,0.00002


## 16. Group Ablation

Domain v1 전체에서 그룹 하나씩 제거합니다.

그룹:
- count pressure
- runner/stretch
- game pressure
- matchup
- recent trend
- sample size
- role

`baseline_no_domain`도 함께 후보로 유지합니다.

In [19]:
# ============================================================
# 16. Group ablation
# ============================================================

GROUP_FEATURE_SET_REGISTRY = {
    "domain_full": list(FEATURES_DOMAIN_FULL),
    "baseline_no_domain": list(BASELINE_FEATURES),
}

for group_name, group_features in DOMAIN_GROUPS.items():
    remove_set = set(group_features)
    GROUP_FEATURE_SET_REGISTRY[f"minus_{group_name}"] = [
        c for c in FEATURES_DOMAIN_FULL
        if c not in remove_set
    ]

group_summaries = [
    domain_full_result["summary"],
    baseline_result["summary"],
]

if RUN_GROUP_ABLATION:
    for experiment_name, feature_list in GROUP_FEATURE_SET_REGISTRY.items():
        if experiment_name in RESULT_REGISTRY:
            continue

        result = run_rolling_oof(
            feature_list,
            experiment_name=experiment_name,
        )
        RESULT_REGISTRY[experiment_name] = result
        group_summaries.append(result["summary"])

group_ablation_results = (
    pd.DataFrame(group_summaries)
    .drop_duplicates("experiment")
    .sort_values("oof_brier")
    .reset_index(drop=True)
)

domain_full_brier = float(
    domain_full_result["summary"]["oof_brier"]
)

group_ablation_results["delta_brier_vs_domain_full"] = (
    group_ablation_results["oof_brier"]
    - domain_full_brier
)

display(
    group_ablation_results[
        [
            "experiment",
            "oof_brier",
            "oof_score",
            "delta_brier_vs_domain_full",
            "std_fold_brier",
            "worst_fold_brier",
            "median_trees_used",
            "n_features",
            "elapsed_sec",
        ]
    ]
)

group_ablation_results.to_csv(
    OUTPUT_DIR / "domain_v1_group_ablation.csv",
    index=False,
)

[minus_count_pressure] valid=2022 | Brier=0.2430631334 | trees=355 | pred_std=0.074076
[minus_count_pressure] valid=2023 | Brier=0.2498911377 | trees=5 | pred_std=0.011730
[minus_count_pressure] valid=2024 | Brier=0.2475907127 | trees=520 | pred_std=0.047891
[minus_runner_stretch] valid=2022 | Brier=0.2430508796 | trees=382 | pred_std=0.074181
[minus_runner_stretch] valid=2023 | Brier=0.2499236669 | trees=4 | pred_std=0.009496
[minus_runner_stretch] valid=2024 | Brier=0.2476148702 | trees=403 | pred_std=0.046985
[minus_game_pressure] valid=2022 | Brier=0.2430542477 | trees=300 | pred_std=0.073765
[minus_game_pressure] valid=2023 | Brier=0.2498922086 | trees=4 | pred_std=0.009196
[minus_game_pressure] valid=2024 | Brier=0.2476014742 | trees=260 | pred_std=0.045490
[minus_matchup] valid=2022 | Brier=0.2430215308 | trees=434 | pred_std=0.075762
[minus_matchup] valid=2023 | Brier=0.2499342311 | trees=5 | pred_std=0.012196
[minus_matchup] valid=2024 | Brier=0.2475818832 | trees=438 | pred_s

,experiment,oof_brier,oof_score,delta_brier_vs_domain_full,std_fold_brier,worst_fold_brier,median_trees_used,n_features,elapsed_sec
0,minus_matchup,0.246844,1253.180406,-0.000021,0.002870,0.249934,434,339,614.488288
1,baseline_no_domain,0.246845,1252.888808,-0.000020,0.002870,0.249933,472,308,543.399646
2,minus_count_pressure,0.246846,1252.133663,-0.000018,0.002837,0.249891,355,331,571.435560
3,minus_game_pressure,0.246847,1251.709201,-0.000017,0.002842,0.249892,260,336,493.473886
4,minus_sample_size,0.246850,1250.574374,-0.000014,0.002872,0.249904,355,339,760.610685
5,minus_recent_trend,0.246854,1249.113103,-0.000010,0.002847,0.249910,352,337,661.199017
6,minus_runner_stretch,0.246861,1246.196983,-0.000003,0.002856,0.249924,382,335,557.709415
7,minus_role,0.246863,1245.580273,-0.000002,0.002858,0.249943,394,331,1107.842688
8,domain_full,0.246864,1244.921045,0.000000,0.002865,0.249941,418,340,625.306375


In [20]:
train.groupby("season")["control_success"].agg(["mean", "size"])

,mean,size
season,,
2019,0.564670,237413
2020,0.532712,244087
2021,0.532762,247088
2022,0.528920,247472
2023,0.499957,245525
2024,0.486105,253507


In [21]:
from sklearn.metrics import roc_auc_score, log_loss

oof = baseline_result["oof_probability"]

rows = []

for season in ROLLING_VALID_SEASONS:
    mask = train["season"].eq(season)

    y = train.loc[mask, TARGET].to_numpy()
    p = oof.loc[mask].to_numpy()

    prevalence = y.mean()
    brier = np.mean((p - y) ** 2)
    oracle_constant_brier = prevalence * (1 - prevalence)

    rows.append({
        "season": season,
        "n": len(y),
        "target_mean": prevalence,
        "pred_mean": p.mean(),
        "pred_std": p.std(),
        "auc": roc_auc_score(y, p),
        "logloss": log_loss(y, p),
        "brier": brier,
        "oracle_constant_brier": oracle_constant_brier,
        "brier_gain_vs_constant":
            oracle_constant_brier - brier,
    })

diagnostic = pd.DataFrame(rows)

display(diagnostic)

,season,n,target_mean,pred_mean,pred_std,auc,logloss,brier,oracle_constant_brier,brier_gain_vs_constant
0,2022,247472,0.528920,0.529118,0.075094,0.581777,0.678832,0.243021,0.249164,0.006143
1,2023,245525,0.499957,0.503210,0.009767,0.530706,0.693014,0.249933,0.250000,0.000067
2,2024,253507,0.486105,0.495157,0.047385,0.553974,0.688297,0.247586,0.249807,0.002221


In [22]:
def calibration_table(y, p, n_bins=10):
    temp = pd.DataFrame({
        "y": y,
        "p": p,
    })

    temp["bin"] = pd.qcut(
        temp["p"],
        q=n_bins,
        duplicates="drop",
    )

    return (
        temp.groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            pred_mean=("p", "mean"),
            actual_rate=("y", "mean"),
        )
        .reset_index()
    )


for season in ROLLING_VALID_SEASONS:
    mask = train["season"].eq(season)

    y = train.loc[mask, TARGET].to_numpy()
    p = baseline_result["oof_probability"].loc[mask].to_numpy()

    print("=" * 70)
    print("SEASON", season)

    display(calibration_table(y, p))

SEASON 2022


,bin,n,pred_mean,actual_rate
0,"(0.323, 0.455]",24748,0.430619,0.430338
1,"(0.455, 0.476]",24747,0.466281,0.468784
2,"(0.476, 0.49]",24747,0.482969,0.478482
3,"(0.49, 0.502]",24748,0.495624,0.488241
4,"(0.502, 0.514]",24746,0.507538,0.502748
5,"(0.514, 0.526]",24747,0.519858,0.520063
6,"(0.526, 0.542]",24747,0.533737,0.532509
7,"(0.542, 0.564]",24747,0.551988,0.551865
8,"(0.564, 0.665]",24747,0.594622,0.599790
9,"(0.665, 0.801]",24748,0.707938,0.716381


SEASON 2023


,bin,n,pred_mean,actual_rate
0,"(0.4949, 0.4957]",25090,0.495280,0.450179
1,"(0.4957, 0.4959]",24392,0.495823,0.461381
2,"(0.4959, 0.4969]",26751,0.496408,0.474450
3,"(0.4969, 0.4981]",21994,0.497424,0.489315
4,"(0.4981, 0.4994]",24562,0.498658,0.489903
5,"(0.4994, 0.5018]",24597,0.500579,0.511932
6,"(0.5018, 0.5049]",25045,0.503634,0.529128
7,"(0.5049, 0.5069]",24094,0.506094,0.552876
8,"(0.5069, 0.5249]",24448,0.509611,0.569903
9,"(0.5249, 0.5413]",24552,0.528805,0.473200


SEASON 2024


,bin,n,pred_mean,actual_rate
0,"(0.317, 0.435]",25351,0.414899,0.406138
1,"(0.435, 0.455]",25351,0.445873,0.437853
2,"(0.455, 0.47]",25350,0.462830,0.451282
3,"(0.47, 0.482]",25351,0.476234,0.467911
4,"(0.482, 0.493]",25351,0.487832,0.479823
5,"(0.493, 0.505]",25350,0.499139,0.487771
6,"(0.505, 0.518]",25351,0.511368,0.502584
7,"(0.518, 0.534]",25350,0.525937,0.512189
8,"(0.534, 0.557]",25351,0.544903,0.532918
9,"(0.557, 0.675]",25351,0.582552,0.582581


In [23]:
def numeric_shift_table(df, season_a, season_b, features):
    rows = []

    for col in features:
        if col not in df.columns:
            continue

        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        a = pd.to_numeric(
            df.loc[df["season"].eq(season_a), col],
            errors="coerce",
        ).dropna()

        b = pd.to_numeric(
            df.loc[df["season"].eq(season_b), col],
            errors="coerce",
        ).dropna()

        if len(a) == 0 or len(b) == 0:
            continue

        pooled_sd = np.sqrt(
            (a.var() + b.var()) / 2
        )

        smd = (
            (b.mean() - a.mean()) / pooled_sd
            if pooled_sd > 0
            else 0
        )

        rows.append({
            "feature": col,
            f"mean_{season_a}": a.mean(),
            f"mean_{season_b}": b.mean(),
            "smd": smd,
            "abs_smd": abs(smd),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("abs_smd", ascending=False)
        .reset_index(drop=True)
    )


shift_22_23 = numeric_shift_table(
    train,
    2022,
    2023,
    BASELINE_FEATURES,
)

display(shift_22_23.head(30))

,feature,mean_2022,mean_2023,smd,abs_smd
0,flat_league_count_flag_success_rate,0.543168,0.539547,-0.480581,0.480581
1,flat_league_count_flag_success_n,94827.223577,126711.883181,0.462727,0.462727
2,flat_league_count_flag_middle_n,94719.245147,126565.719633,0.462309,0.462309
3,flat_league_count_flag_offspeed_n,94719.245147,126565.719633,0.462309,0.462309
4,flat_league_count_flag_reverse_n,94719.245147,126565.719633,0.462309,0.462309
5,flat_league_count_flag_ball_n,94719.245147,126565.719633,0.462309,0.462309
6,flat_league_count_flag_strike_n,94719.245147,126565.719633,0.462309,0.462309
7,flat_league_count_flag_fastball_n,94719.245147,126565.719633,0.462309,0.462309
8,flat_league_count_flag_breaking_n,94719.245147,126565.719633,0.462309,0.462309
9,flat_league_count_hand_flag_success_n,47410.865314,63530.084704,0.461786,0.461786


In [24]:
CHECK_COLS = [
    "asof_pitcher_success_rate",
    "asof_pitcher_middle_rate",
    "asof_pitcher_reverse_rate",
    "asof_pitcher_ball_rate",
    "asof_pitcher_strike_rate",

    "asof_pitcher_fastball_rate",
    "asof_pitcher_breaking_rate",
    "asof_pitcher_offspeed_rate",

    "asof_pitcher_n",

    "asof_pitcher_prev1_game_success_rate",
    "asof_pitcher_prev3_game_success_rate",
    "asof_pitcher_prev5_game_success_rate",

    "inning",
    "balls_before",
    "strikes_before",
    "outs_before",

    "li",
]

display(
    numeric_shift_table(
        train,
        2022,
        2023,
        CHECK_COLS,
    )
)

,feature,mean_2022,mean_2023,smd,abs_smd
0,asof_pitcher_n,3194.707236,3880.774377,0.238302,0.238302
1,asof_pitcher_prev5_game_success_rate,0.525919,0.508793,-0.210614,0.210614
2,asof_pitcher_success_rate,0.535402,0.523730,-0.209247,0.209247
3,asof_pitcher_prev3_game_success_rate,0.526313,0.507353,-0.203504,0.203504
4,asof_pitcher_reverse_rate,0.223366,0.233023,0.169273,0.169273
5,asof_pitcher_prev1_game_success_rate,0.526797,0.504433,-0.161356,0.161356
6,asof_pitcher_middle_rate,0.143750,0.147506,0.146969,0.146969
7,asof_pitcher_fastball_rate,0.555407,0.545318,-0.108168,0.108168
8,asof_pitcher_strike_rate,0.442170,0.444210,0.058139,0.058139
9,asof_pitcher_breaking_rate,0.289721,0.295620,0.057050,0.057050


In [ ]:
def pitcher_population_stats(df):
    seasons = sorted(df["season"].unique())

    rows = []

    for season in seasons:
        current_pitchers = set(
            df.loc[
                df["season"].eq(season),
                "pitcher_id"
            ].dropna()
        )

        previous_pitchers = set(
            df.loc[
                df["season"].lt(season),
                "pitcher_id"
            ].dropna()
        )

        seen = current_pitchers & previous_pitchers
        new = current_pitchers - previous_pitchers

        current_rows = df["season"].eq(season)

        new_pitcher_rows = (
            current_rows
            & ~df["pitcher_id"].isin(previous_pitchers)
        )

        rows.append({
            "season": season,
            "n_pitchers": len(current_pitchers),
            "seen_pitchers": len(seen),
            "new_pitchers": len(new),
            "new_pitcher_share":
                len(new) / len(current_pitchers),
            "new_pitcher_row_share":
                new_pitcher_rows.sum()
                / current_rows.sum(),
        })

    return pd.DataFrame(rows)


display(pitcher_population_stats(train_raw))

,season,n_pitchers,seen_pitchers,new_pitchers,new_pitcher_share,new_pitcher_row_share
0,2019,355,0,355,1.000000,1.000000
1,2020,356,246,110,0.308989,0.223777
2,2021,386,291,95,0.246114,0.205672
3,2022,390,302,88,0.225641,0.157392
4,2023,382,319,63,0.164921,0.137982
5,2024,391,310,81,0.207161,0.198606


: 

In [ ]:
# ============================================================
# 17. Select group-level feature set conservatively
# ============================================================

best_group_row = group_ablation_results.iloc[0]
best_group_name = str(best_group_row["experiment"])
best_group_brier = float(best_group_row["oof_brier"])

# domain_full보다 좋아도 아주 작은 차이면 full 유지
if (
    best_group_name != "domain_full"
    and (domain_full_brier - best_group_brier) < ABLATION_IMPROVEMENT_TOL
):
    BEST_GROUP_EXPERIMENT = "domain_full"
else:
    BEST_GROUP_EXPERIMENT = best_group_name

BEST_GROUP_FEATURES = list(
    RESULT_REGISTRY[BEST_GROUP_EXPERIMENT]["features"]
)
BEST_GROUP_RESULT = RESULT_REGISTRY[BEST_GROUP_EXPERIMENT]

print(
    f"Selected group-level feature set: {BEST_GROUP_EXPERIMENT}\n"
    f"  n_features={len(BEST_GROUP_FEATURES)}\n"
    f"  OOF Brier={BEST_GROUP_RESULT['summary']['oof_brier']:.10f}"
)

## 18. Individual Ablation

Group ablation에서 살아남은 Domain v1 피처만 대상으로 하나씩 제거합니다.

- 기준: `BEST_GROUP_FEATURES`
- 각 후보: `BEST_GROUP_FEATURES - {feature}`
- 모든 비교는 동일한 rolling folds

그 뒤, **개별 제거 시 실제로 개선된 피처들**을 한꺼번에 제거한 combined candidate를 한 번 더 rolling CV로 검증합니다.

In [ ]:
# ============================================================
# 18. Individual leave-one-feature-out ablation
# ============================================================

surviving_domain_features = [
    c for c in DOMAIN_FEATURES
    if c in BEST_GROUP_FEATURES
]

individual_rows = [
    {
        **BEST_GROUP_RESULT["summary"],
        "removed_feature": None,
        "delta_brier_vs_group_selected": 0.0,
    }
]

INDIVIDUAL_RESULT_REGISTRY = {}

if RUN_INDIVIDUAL_ABLATION and surviving_domain_features:
    valid_seasons_for_individual = (
        ROLLING_VALID_SEASONS
        if FULL_ROLLING_INDIVIDUAL_ABLATION
        else [ROLLING_VALID_SEASONS[-1]]
    )

    for feature in surviving_domain_features:
        experiment_name = f"minus_feature__{feature}"
        feature_list = [
            c for c in BEST_GROUP_FEATURES
            if c != feature
        ]

        result = run_rolling_oof(
            feature_list,
            experiment_name=experiment_name,
            valid_seasons=valid_seasons_for_individual,
        )

        INDIVIDUAL_RESULT_REGISTRY[feature] = result

        row = dict(result["summary"])
        row["removed_feature"] = feature

        # full 3-fold일 때만 group-selected OOF와 직접 delta 비교
        if FULL_ROLLING_INDIVIDUAL_ABLATION:
            row["delta_brier_vs_group_selected"] = (
                float(row["oof_brier"])
                - float(BEST_GROUP_RESULT["summary"]["oof_brier"])
            )
        else:
            row["delta_brier_vs_group_selected"] = np.nan

        individual_rows.append(row)

individual_ablation_results = (
    pd.DataFrame(individual_rows)
    .sort_values("oof_brier")
    .reset_index(drop=True)
)

display(
    individual_ablation_results[
        [
            "removed_feature",
            "experiment",
            "oof_brier",
            "oof_score",
            "delta_brier_vs_group_selected",
            "std_fold_brier",
            "worst_fold_brier",
            "median_trees_used",
            "n_features",
        ]
    ]
)

individual_ablation_results.to_csv(
    OUTPUT_DIR / "domain_v1_individual_ablation.csv",
    index=False,
)

In [ ]:
# ============================================================
# 19. Combined individual pruning + final selection
# ============================================================

FINAL_SELECTION_EXPERIMENT = BEST_GROUP_EXPERIMENT
FINAL_SELECTION_RESULT = BEST_GROUP_RESULT
FINAL_FEATURES = list(BEST_GROUP_FEATURES)

INDIVIDUALLY_PRUNED_FEATURES = []
combined_prune_result = None

if (
    RUN_INDIVIDUAL_ABLATION
    and FULL_ROLLING_INDIVIDUAL_ABLATION
    and surviving_domain_features
):
    group_brier = float(BEST_GROUP_RESULT["summary"]["oof_brier"])

    for _, row in individual_ablation_results.iterrows():
        feature = row.get("removed_feature")
        if pd.isna(feature) or feature is None:
            continue

        improvement = group_brier - float(row["oof_brier"])
        if improvement > INDIVIDUAL_IMPROVEMENT_TOL:
            INDIVIDUALLY_PRUNED_FEATURES.append(str(feature))

    print("Individually harmful candidates:", INDIVIDUALLY_PRUNED_FEATURES)

    if INDIVIDUALLY_PRUNED_FEATURES:
        combined_features = [
            c for c in BEST_GROUP_FEATURES
            if c not in set(INDIVIDUALLY_PRUNED_FEATURES)
        ]

        combined_prune_result = run_rolling_oof(
            combined_features,
            experiment_name="combined_individual_prune",
        )

        RESULT_REGISTRY["combined_individual_prune"] = combined_prune_result

        combined_brier = float(
            combined_prune_result["summary"]["oof_brier"]
        )

        if (
            group_brier - combined_brier
        ) > INDIVIDUAL_IMPROVEMENT_TOL:
            FINAL_SELECTION_EXPERIMENT = "combined_individual_prune"
            FINAL_SELECTION_RESULT = combined_prune_result
            FINAL_FEATURES = list(combined_features)

print()
print("FINAL SELECTION")
print("  experiment :", FINAL_SELECTION_EXPERIMENT)
print("  n_features :", len(FINAL_FEATURES))
print(
    "  OOF Brier :",
    f"{FINAL_SELECTION_RESULT['summary']['oof_brier']:.10f}"
)
print(
    "  fold std  :",
    f"{FINAL_SELECTION_RESULT['summary']['std_fold_brier']:.10f}"
)
print(
    "  worst fold:",
    f"{FINAL_SELECTION_RESULT['summary']['worst_fold_brier']:.10f}"
)

## 20. 최종 재학습

선택된 feature set으로 전체 train을 재학습합니다.

iterations는 선택된 rolling CV의 `median_trees_used`를 사용합니다.

In [ ]:
# ============================================================
# 20. Final retraining
# ============================================================

FINAL_CAT_COLS = [
    c for c in CAT_COLS
    if c in FINAL_FEATURES
]

final_iterations = max(
    50,
    int(FINAL_SELECTION_RESULT["summary"]["median_trees_used"]),
)

final_params = dict(BASE_CATBOOST_PARAMS)
final_params["iterations"] = final_iterations
final_params["verbose"] = 50

print("Final parameters")
print("  selected experiment :", FINAL_SELECTION_EXPERIMENT)
print("  features            :", len(FINAL_FEATURES))
print("  categorical         :", len(FINAL_CAT_COLS))
print("  iterations          :", final_iterations)

final_model = CatBoostClassifier(**final_params)

final_model.fit(
    train[FINAL_FEATURES],
    train[TARGET],
    cat_features=FINAL_CAT_COLS,
)

print("Final full-data retraining complete.")

## 21. 모델 / preprocessing artifact 저장

저장 파일:

- `catboost_domain_v1.cbm`
- `model_bundle_domain_v1.pkl`
- `history_features_domain_v1.pkl`
- `pitcher_role_lookup_domain_v1.pkl`
- `pitcher_first_season_domain_v1.pkl`
- `domain_v1_metadata.json`
- group / individual ablation CSV

`history_features_domain_v1.pkl`은 이후 test inference에서
strict historical Flat/Current feature를 재현하기 위한 compact history입니다.

In [ ]:
# ============================================================
# 21. Save model + artifacts
# ============================================================

MODEL_CBM_PATH = OUTPUT_DIR / "catboost_domain_v1.cbm"
MODEL_BUNDLE_PATH = OUTPUT_DIR / "model_bundle_domain_v1.pkl"
HISTORY_PATH = OUTPUT_DIR / "history_features_domain_v1.pkl"
ROLE_LOOKUP_PATH = OUTPUT_DIR / "pitcher_role_lookup_domain_v1.pkl"
FIRST_SEASON_PATH = OUTPUT_DIR / "pitcher_first_season_domain_v1.pkl"
METADATA_PATH = OUTPUT_DIR / "domain_v1_metadata.json"

final_model.save_model(str(MODEL_CBM_PATH))

# test 2025 등 미래 시즌 inference용: 전체 train으로 만든 role lookup
final_role_lookup = make_role_lookup(history_compact)

joblib.dump(
    history_compact,
    HISTORY_PATH,
    compress=3,
)
joblib.dump(
    final_role_lookup,
    ROLE_LOOKUP_PATH,
    compress=3,
)
joblib.dump(
    pitcher_first_season,
    FIRST_SEASON_PATH,
    compress=3,
)

selected_domain_features = [
    c for c in FINAL_FEATURES
    if c in DOMAIN_FEATURES
]

selected_domain_groups = {
    group_name: [
        c for c in group_features
        if c in selected_domain_features
    ]
    for group_name, group_features in DOMAIN_GROUPS.items()
}

bundle = {
    "catboost_model": final_model,
    "features": list(FINAL_FEATURES),
    "cat_cols": list(FINAL_CAT_COLS),
    "iterations": int(final_iterations),
    "final_params": dict(final_params),
    "id_column": ID,
    "target_column": TARGET,
    "feature_version": "0816_domain_v1",
    "selected_feature_set": FINAL_SELECTION_EXPERIMENT,
    "selected_domain_features": list(selected_domain_features),
    "selected_domain_groups": selected_domain_groups,
    "rolling_valid_seasons": list(ROLLING_VALID_SEASONS),
    "oof_summary": dict(FINAL_SELECTION_RESULT["summary"]),
    "flat_alpha": float(FLAT_ALPHA),
    "context_effect_alpha": float(CONTEXT_EFFECT_ALPHA),
    "current_season_alpha": float(CURRENT_SEASON_ALPHA),
    "logit_eps": float(LOGIT_EPS),
    "role_config": {
        "min_pitches": int(ROLE_MIN_PITCHES),
        "starter_early_share": float(ROLE_STARTER_EARLY_SHARE),
        "starter_mean_inning_max": float(ROLE_STARTER_MEAN_INNING_MAX),
        "closer_late_share": float(ROLE_CLOSER_LATE_SHARE),
    },
    "artifact_files": {
        "model_cbm": MODEL_CBM_PATH.name,
        "history": HISTORY_PATH.name,
        "role_lookup": ROLE_LOOKUP_PATH.name,
        "first_season": FIRST_SEASON_PATH.name,
        "metadata": METADATA_PATH.name,
    },
    "notes": {
        "trackman_history_used": False,
        "batter_flat_used": False,
        "latent_skill_used": False,
        "latent_reason": (
            "0816 SOTA inference notebook does not provide a fold-safe "
            "cross-fit latent-skill training recipe; intentionally excluded "
            "from rolling CV to avoid leakage."
        ),
    },
}

joblib.dump(
    bundle,
    MODEL_BUNDLE_PATH,
    compress=3,
)

metadata = {
    "feature_version": "0816_domain_v1",
    "selected_feature_set": FINAL_SELECTION_EXPERIMENT,
    "n_features": len(FINAL_FEATURES),
    "features": list(FINAL_FEATURES),
    "cat_cols": list(FINAL_CAT_COLS),
    "selected_domain_features": list(selected_domain_features),
    "selected_domain_groups": selected_domain_groups,
    "rolling_valid_seasons": list(ROLLING_VALID_SEASONS),
    "oof_summary": {
        k: (
            float(v)
            if isinstance(v, (np.floating, float))
            else int(v)
            if isinstance(v, (np.integer, int))
            else v
        )
        for k, v in FINAL_SELECTION_RESULT["summary"].items()
    },
    "iterations": int(final_iterations),
    "flat_alpha": float(FLAT_ALPHA),
    "context_effect_alpha": float(CONTEXT_EFFECT_ALPHA),
    "current_season_alpha": float(CURRENT_SEASON_ALPHA),
    "logit_eps": float(LOGIT_EPS),
    "individually_pruned_features": list(INDIVIDUALLY_PRUNED_FEATURES),
    "trackman_history_used": False,
    "batter_flat_used": False,
    "latent_skill_used": False,
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved:")
for path in [
    MODEL_CBM_PATH,
    MODEL_BUNDLE_PATH,
    HISTORY_PATH,
    ROLE_LOOKUP_PATH,
    FIRST_SEASON_PATH,
    METADATA_PATH,
    OUTPUT_DIR / "domain_v1_group_ablation.csv",
    OUTPUT_DIR / "domain_v1_individual_ablation.csv",
]:
    print(" ", path)

## 22. 결과 해석 체크리스트

실행 후 아래 순서로 확인합니다.

1. `baseline_no_domain` vs `domain_full`
   - Domain 전체가 실제로 Brier를 낮추는지
2. Group ablation
   - `minus_runner_stretch`, `minus_role` 등에서 Brier가 악화되면 해당 그룹이 유효
   - Brier뿐 아니라 `worst_fold_brier`, `std_fold_brier`도 확인
3. Individual ablation
   - 제거했을 때 Brier가 개선되는 피처는 harmful candidate
4. Combined prune
   - 개별 harmful feature를 함께 뺐을 때도 실제로 개선되는지 최종 재확인
5. 최종 모델
   - rolling fold median trees로 전체 train 재학습
6. 추론 노트북 제작 시
   - 본 notebook의 `add_base_features`
   - `add_domain_static_features`
   - Flat/Current/context/HLogit
   - 저장된 history/role lookup
   을 동일하게 재현해야 함

### Latent skill

이 버전에서 일부러 제외했습니다.  
다음 버전에서 추가하려면 각 rolling fold마다 skill model도 다음처럼 학습해야 합니다.

`skill train = season < valid_season`

그 뒤 validation season에 skill prediction을 만들고 메인 CatBoost에 넣어야 합니다.